Para este ejercicio, implementaremos un flujo completo de clasificación utilizando Regresión Logística frente a Random Forest. El objetivo será predecir el tipo de tumor (maligno o benigno) utilizando el conjunto de datos estándar Breast Cancer de scikit-learn.

Se aplicará Grid Search con validación cruzada (k-fold = 5) para el ajuste de hiperparámetros. Las métricas estandarizadas a registrar en MLflow serán Exactitud (Accuracy), Precisión (Precision), Sensibilidad (Recall) y Puntuación F1 (F1-Score).

# Descargar Dependencias

In [ ]:
pip install mlflow scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Cargar y preparar los datos

In [ ]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Definir la función para métricas estandarizadas

In [ ]:
def calcular_metricas(y_real, y_pred):
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred),
        "recall": recall_score(y_real, y_pred),
        "f1_score": f1_score(y_real, y_pred)
    }

# Configurar con MLflow

In [ ]:
mlflow.set_experiment("Clasificacion_Cancer_Mama_Comparativa")

2026/10/07 17:42:29 INFO mlflow.tracking.fluent: Experiment with name 'Clasificacion_Cancer_Mama_Comparativa' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/hemke/Documents/Master - Inteligencia Artificial/Gestión de '
 'proyectos de Inteligencia Artificial/Actividad '
 '5/Actividad-5/fuentes/mlruns/2'), creation_time=1791416549061, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1791416549061, lifecycle_stage='active', name='Clasificacion_Cancer_Mama_Comparativa', tags={}, trace_location=None, workspace='default'>

# MODELO 1: REGRESIÓN LOGÍSTICA

In [ ]:
with mlflow.start_run(run_name="Regresion_Logistica_GridSearch"):
    print("Entrenando Regresión Logística...")

    # Definir modelo y espacio de hiperparámetros
    lr = LogisticRegression(max_iter=10000, random_state=42)
    param_grid_lr = {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['lbfgs', 'liblinear']
    }

    # Ejecutar Grid Search con Validación Cruzada (k-fold=5)
    grid_lr = GridSearchCV(lr, param_grid_lr, cv=5, scoring='f1', n_jobs=-1)
    grid_lr.fit(X_train, y_train)

    # Obtener el mejor modelo y predecir
    best_lr = grid_lr.best_estimator_
    y_pred_lr = best_lr.predict(X_test)
    metricas_lr = calcular_metricas(y_test, y_pred_lr)

    # Registrar en MLflow
    mlflow.log_params(grid_lr.best_params_)
    mlflow.log_metrics(metricas_lr)
    mlflow.sklearn.log_model(best_lr, "modelo_logistico")

    print(f"Mejores parámetros RL: {grid_lr.best_params_}")
    print(f"F1-Score RL: {metricas_lr['f1_score']:.4f}\n")

Entrenando Regresión Logística...


2026/10/07 17:42:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Mejores parámetros RL: {'C': 10, 'solver': 'liblinear'}
F1-Score RL: 0.9655



# MODELO 2: RANDOM FOREST

In [ ]:
with mlflow.start_run(run_name="Random_Forest_GridSearch"):
    print("Entrenando Random Forest...")

    # Definir modelo y espacio de hiperparámetros
    rf = RandomForestClassifier(random_state=42)
    param_grid_rf = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }

    # Ejecutar Grid Search con Validación Cruzada (k-fold=5)
    grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
    grid_rf.fit(X_train, y_train)

    # Obtener el mejor modelo y predecir
    best_rf = grid_rf.best_estimator_
    y_pred_rf = best_rf.predict(X_test)
    metricas_rf = calcular_metricas(y_test, y_pred_rf)

    # Registrar en MLflow
    mlflow.log_params(grid_rf.best_params_)
    mlflow.log_metrics(metricas_rf)
    mlflow.sklearn.log_model(
      best_rf,
      "modelo_random_forest",
      skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print(f"Mejores parámetros RF: {grid_rf.best_params_}")
    print(f"F1-Score RF: {metricas_rf['f1_score']:.4f}")

Entrenando Random Forest...


2026/10/07 17:43:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Mejores parámetros RF: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
F1-Score RF: 0.9722


# Visualización

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Cargar los datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# Configurar el estilo de las gráficas
sns.set_theme(style="whitegrid")

# ==========================================
# Gráfica 1: Distribución de Clases
# ==========================================
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='target', data=df, palette='viridis')
plt.title('Distribución de Clases (0: Maligno, 1: Benigno)')
plt.xlabel('Tipo de Tumor')
plt.ylabel('Cantidad')

# Agregar los valores numéricos sobre las barras
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')
plt.tight_layout()
plt.show()

# ==========================================
# Gráfica 2: Matriz de Correlación (Características 'mean')
# ==========================================
# Seleccionamos solo las primeras 10 variables para que la gráfica sea legible en tu PDF
columnas_mean = list(df.columns[:10]) + ['target']
matriz_corr = df[columnas_mean].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
plt.title('Matriz de Correlación (Características Medias y Target)')
plt.tight_layout()
plt.show()